In [1]:
from IPython.display import Image

- cv2.distanceTransform：是计算二值图像中每一个非零像素点（通常代表前景或目标对象）到距离它最近的零像素点（通常代表背景）的空间距离。
    - $D(p)=\min_{q\in Z} d(p,q)$
        - $p$: 当前像素
        - $Z$: 是所有值为 0 的像素集合
        - $d(p,q)$: 像素 $p$ 到 $q$ 的距离
    - distanceType 可以理解成“怎么定义距离”：
        - cv2.DIST_L1：曼哈顿距离，像城市棋盘路网，只能横着走或竖着走。
            - $d_{L1}((x_1,y_1),(x_2,y_2)) = |x_1-x_2|+|y_1-y_2|$
        - cv2.DIST_L2：欧氏距离
            - $d_{L2}((x_1,y_1),(x_2,y_2)) = \sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$
        - cv2.DIST_C：棋盘/切比雪夫风格的距离
            - $d_{\infty}((x_1,y_1),(x_2,y_2))=\max(|x_1-x_2|,|y_1-y_2|)$
            - “国王步数”：如同国际象棋中的国王，无论是横、竖还是斜向移动一步，都被计为等价的距离 1。

### 二值化

```
image = Image.open(input_path).convert("RGB")
gray = np.array(image.convert("L"))
open_mask = (gray > 200).astype(np.uint8)
```

### fillPoly vs. fillConvexPoly

> 两个函数的核心职责都是将二维平面上由离散顶点围成的封闭区域用指定颜色填满
- cv2.fillPoly()：全能型拓扑大师，作用：用于填充一个或多个任意形状的多边形。
- cv2.fillConvexPoly()：极速特化型选手，专门且仅用于填充单个凸多边形。
    - 一个多边形（或点集） $S$ 被称为凸的，当且仅当在该集合内任取两点，连接这两点的线段上的所有点都必然属于该集合。用数学公式表述为：$$\forall x, y \in S, \quad \forall t \in [0, 1], \quad tx + (1-t)y \in S$$这意味着凸多边形的边缘不会向内“凹陷”。

### 形态学运算

> cv2.morphologyEx

In [3]:
Image(url='https://resize-image.vocus.cc/resize?compression=6&norotation=true&url=https%3A%2F%2Fimages.vocus.cc%2F83e09494-0f07-40aa-9412-921a9b7255ad.png&width=740&sign=MNTf_zX-uJu26ww3E5PTQG5DRP5wDEZp3hAHRyhSTEY', 
      width=500)

- 基础
    - 腐蚀 (Erosion)
    - 膨胀 (Dilation)
- 复合
    - MORPH_CLOSE: 闭运算
        - 去噪；
        - 先膨胀，后腐蚀；
    - MORPH_OPEN: 开运算
    - MORPH_TOPHAT
    - MORPH_BLACKHAT

- app
    - MORPH_CLOSE (去掉小裂缝之后) => 再进行连通性分析 

### 连通性分析

- `cv2.connectedComponentsWithStats`
    - 8 联通，4 联通

In [4]:
import cv2
import numpy as np

# 1. 构造 5x5 二值矩阵 (极简测试场)
img = np.array([
    [255,   0,   0,   0,   0],
    [  0, 255,   0,   0,   0],
    [  0,   0,   0, 255, 255],
    [  0,   0,   0, 255, 255],
    [  0,   0,   0,   0,   0]
], dtype=np.uint8)

# 2. 分别应用 4联通 与 8联通 的连通域分析
num_4, labels_4, _, _ = cv2.connectedComponentsWithStats(img, connectivity=4)
num_8, labels_8, _, _ = cv2.connectedComponentsWithStats(img, connectivity=8)

In [7]:
# 1 个背景区和 3 个独立前景物体
labels_4

array([[1, 0, 0, 0, 0],
       [0, 2, 0, 0, 0],
       [0, 0, 0, 3, 3],
       [0, 0, 0, 3, 3],
       [0, 0, 0, 0, 0]], dtype=int32)

In [9]:
# 1 个背景区和 2 个独立前景物体
labels_8

array([[1, 0, 0, 0, 0],
       [0, 1, 0, 0, 0],
       [0, 0, 0, 2, 2],
       [0, 0, 0, 2, 2],
       [0, 0, 0, 0, 0]], dtype=int32)

### perspective transform

> 相机小孔成像投影到 rgb 2d，因为有 perspective 的现象（近大远小），原来是直线 rgb 2d 上也会是直线，但原来的平行线，会在 rgb 2d 上相交；

- 现实世界图像时，往往首先遭遇“维度降维”带来的信息畸变。当镜头捕获三维物理世界时，平行的铁轨在视平线上不可逆转地交汇于一点（消失点）。透视变换（Perspective Transformation），正是描述并尝试在二维图像内部“模拟并逆转”这一三维投影规律的数学杠杆。
    - `cv2.getPerspectiveTransform`: image to rect
        - Homography Matrix：两个平面之间的二维射影几何变换。
        - 你给它 4 个畸变的源点（梯形）和 4 个目标点（完美矩形），它就能解算出一个 $3\times 3$ 的矩阵（单应性矩阵 $H$）。
        - 矩阵的意义：只要把原始图像上的任何一个像素坐标乘以这个矩阵 $H$，就能算出它在“展平”后的图像上的新位置。我们同时计算了正向（原图到展平图）和逆向（展平图回原图）的矩阵，以便后续随时在两个空间中穿梭。
    - `cv2.warpPerspective`: rect to image
- 透视变换的推导有一个不可撼动的前提：“直线投影后必须还是直线”。然而，现实相机的镜头往往存在桶形或枕形畸变（Distortion），这就导致图像边缘的直线已经发生物理层面的弯曲。如果研究者越过相机标定（Camera Calibration）与去畸变（Undistort）的步骤，直接使用四个点计算矩阵，将会在图像边缘引发无法弥合的错位撕裂。